# 🏆 Modul 06 Hands-on: Production Computer Vision System

Di modul puncak track Computer Vision ini, Anda akan membangun sistem klasifikasi citra end-to-end:
1. **Pipeline Augmentasi Produksi**.
2. **Training & Validation Engine** lengkap dengan metrik presisi per kelas.
3. **TorchScript Serialization**: Meng-export model ke format C++ runtime (`torch.jit.trace`).
4. **Stand-alone Inference Runner** yang siap dipanggil oleh sistem eksternal.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights
import numpy as np
import time, os

device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"🚀 Engine aktif: {device}")

## 1. Membangun Model Ringan Produksi (MobileNetV3)

In [ ]:
weights = MobileNet_V3_Small_Weights.DEFAULT
model = mobilenet_v3_small(weights=weights)
# Ganti classifier untuk 4 kategori kustom
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 4)
model = model.to(device)
model.eval()
print("Model MobileNetV3 siap digunakan!")

## 2. Export ke TorchScript untuk Deployment Production
TorchScript memungkinkan model dieksekusi di server tanpa dependensi Python!

In [ ]:
dummy_input = torch.randn(1, 3, 224, 224).to(device)
traced_model = torch.jit.trace(model, dummy_input)

TS_PATH = "cv_classifier_traced.pt"
traced_model.save(TS_PATH)
print(f"✅ Model berhasil di-export ke: {TS_PATH} ({os.path.getsize(TS_PATH)/1024/1024:.2f} MB)")

## 3. Benchmarking Inferensi Cepat

In [ ]:
loaded_ts = torch.jit.load(TS_PATH, map_location=device)
loaded_ts.eval()

# Warmup
for _ in range(10): _ = loaded_ts(dummy_input)

N_RUNS = 200
start = time.time()
with torch.no_grad():
    for _ in range(N_RUNS):
        _ = loaded_ts(dummy_input)
total = time.time() - start

print(f"⏱️ Rata-rata Latensi Inferensi: {(total/N_RUNS)*1000:.2f} ms per frame")
print(f"⚡ Throughput: {N_RUNS/total:.1f} FPS (Siap untuk streaming video real-time!)")